# A Voice That Understands Me
## Team 4 | English, Kenyan English and Kiswahili | AI for Accessibility Hackathon, Nairobi

**Author narrative: Christopher Nguu Kioko, Team 4 | Research notebook | 2 November 2026**

> **My objective.** I am developing and evaluating a personalized, opt-in voice command adapter for a person whose speech may not be correctly understood by a conventional recognizer. I will compare generic transcription, Kenyan atypical speech transcription, fixed pretrained audio representations and carefully controlled personalization. I will connect the resulting *accepted* utterances to the official AI4A11y Controller, not replace its adaptation or permission model.

**Integrity note:** I distinguish my research plan from work I actually ran. A code cell does not imply a model has been trained, an evaluation conducted or a co-designer has agreed to participate. I will not publish private audio, pretend to have accessed gated corpora, or report synthetic test results as accessibility performance.

**Key official references:** [Toolkit repository](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit), [Controller design](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit/blob/main/controller/DESIGN.md), [Voice mode](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit/blob/main/docs/voice-mode.md), [ControlPort protocol](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit/blob/main/controller/PROTOCOL.md), [Controller implementation](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit/blob/main/controller/createController.js).

**Notebook design:** data provenance → ethical enrollment → reproducible manifest → audio QA → leakage-safe split → ASR baselines → encoder embeddings → prototypes and discriminative model → threshold selection → selective encoder fine tuning → untouched test evaluation → ablations and error analysis → artifact export → Python inference service → browser/Toolkit bridge → reproducibility and submission checks.

## 0. Research questions and falsifiable hypotheses

**H1:** The person-specific embedding or classifier approach improves accepted-command recognition compared with unadapted transcription on a separate recording session, without increasing false activations. **H2:** Kenyan atypical speech-adapted ASR provides a stronger baseline than ordinary Whisper on these languages, but this is to be measured on our task. **H3:** Selective encoder fine tuning improves over a frozen encoder only when enough diverse examples exist. **H4:** The app can be operated by the co-designer with their preferred listening initiation, feedback and cancellation.

**Primary outcome:** task completion by the co-designer. **Safety guardrail:** unintended actions and explicit unknown-speech rejection. **Secondary:** accepted-command recall, macro F1, latency, session robustness, English vs Kiswahili parity. **No arbitrary 85% accuracy promise:** targets are engineering hypotheses, not claims of success.

The co-designer chooses the useful commands and may decline recording, stop, request deletion, or opt for nonvoice control. Their impairment or diagnosis is not a model input or an eligibility requirement.

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd
import numpy as np
# Initial imports support the source registry before experiment configuration.

## 1. Verified data and model resource registry

| Resource | Geography / content | Role | Access / compliance | URL |
|:---|:---|:---|:---|:---|
| **CDLI Kenyan English nonstandard v1.0** | 32.3h, 5,998 samples, 52 Kenyan speakers with speech impairments | Relevant research/training only when approved | Gated, CC BY 4.0 displayed, **additional terms / approval** | [Dataset](https://huggingface.co/datasets/cdli/kenyan_english_nonstandard_speech_v1.0) |
| **CDLI Kenyan Swahili nonstandard v1.0** | 32.5h, 5,535 samples, 52 Kenyan speakers with speech impairments | Same | Gated, additional terms / approval | [Dataset](https://huggingface.co/datasets/cdli/kenyan_swahili_nonstandard_speech_v1.0) |
| **CDLI Whisper small Kenyan English** | Fine tuned with atypical Kenyan English; publisher WER .21 versus .26 generic | Language-specific ASR baseline | Model terms and privacy must be reviewed | [Model](https://huggingface.co/cdli/whisper-small_finetuned_kenyan_english_nonstandard_speech_v1.0) |
| **CDLI Whisper small Kenyan Swahili** | Fine tuned with atypical Kenyan Swahili; publisher WER .44 versus .57 unadapted Swahili model | Language-specific ASR baseline | Model terms and privacy must be reviewed | [Model](https://huggingface.co/cdli/whisper-small_finetuned_kenyan_swahili_nonstandard_speech_v1.0) |
| **AfriSpeech 200** | African-accented English including Kenyan speakers | Accent stress testing, with proper split | **CC BY NC SA 4.0**, not interchangeable with atypical speech | [Dataset](https://huggingface.co/datasets/intronhealth/afrispeech-200) |
| **FLEURS** | Multilingual read speech, language-specific configs | Typical-speech Kiswahili external baseline | CC BY 4.0; inspect config and region rather than assume Kenyan accent | [Dataset](https://huggingface.co/datasets/google/fleurs) |
| **OpenSLR 25 ALFFA** | Swahili broadcast news and transcripts | Typical-speech Swahili robustness | MIT, domain mismatch | [Dataset](https://www.openslr.org/25/) |
| **Mozilla Common Voice via MDC** | Current Kiswahili and English releases | Typical-speech robustness | Current distribution from Mozilla Data Collective; verify exact release | [Catalog](https://mozilladatacollective.com/) |
| **Google Speech Commands** | Typical short English commands | Pipeline smoke tests and negatives only | Review dataset release terms | [Dataset](https://huggingface.co/datasets/google/speech_commands) |
| **XLS-R 300M** | Multilingual SSL speech encoder, 128 languages | Main embedding/transfer-learning candidate | Heavy model; requires compute and model license review | [Model](https://huggingface.co/facebook/wav2vec2-xls-r-300m) |
| **Wav2Vec2 Base** | General pretrained English-centric speech encoder | Lighter encoder comparator | English skew; not guaranteed for Kiswahili | [Model](https://huggingface.co/facebook/wav2vec2-base) |

**Critical research distinctions:**

1. The two CDLI corpora appear to describe the same 52-speaker cohort. I will **not** represent these as 104 independent speakers or pair records across languages without a documented ID mapping and explicit authorization.
2. Published WER is **not** our command accuracy and cannot be carried into our task results.
3. Generic Kiswahili data does not solve atypical speech or personalized commands on its own.
4. A source may be openly listed yet still require a gated approval, contributor consent obligations, nonreidentification and restrictions on derivative distribution.
5. **Dataset license is not model license**. I log approvals, allowed uses, files, versions and deletion obligations independently.
6. No code here automatically fetches gated CDLI recordings, and no code uses official CDLI test splits for training.

Research background: [Yee et al. 2023 phrase matching](https://www.isca-archive.org/interspeech_2023/yee23_interspeech.html), [2024 atypical speech parameter-efficient personalization](https://aclanthology.org/2024.tacl-1.65/), [2025 atypical speech modeling](https://aclanthology.org/2025.emnlp-main.1701/).

**2026 compatibility warning:** The legacy `google/speech_commands` Hugging Face entry currently reports a dataset-script compatibility failure with modern `datasets`. I use it as a documented data source, not as an assumed one-line `load_dataset` dependency. Download from the [TensorFlow speech commands source](https://www.tensorflow.org/datasets/catalog/speech_commands) or the repository's raw release after verifying its conditions, and normalize locally to the explicit manifest schema. The same principle applies to any aging loader.


### 1A. CDLI gated corpus access: a controlled intake process

I will ask for approval to access **both** CDLI datasets and read the [terms and conditions](https://huggingface.co/datasets/cdli/terms_and_conditions) before transferring any recording or fine tuning any model. The dataset descriptions explicitly identify speaker-disjoint and phrase-disjoint train/dev/test partitions; I preserve them exactly. The release contains transcribed utterances, **not** guaranteed repeated `NEXT`, `BACK`, `UNDO` enrollment examples. This corpus is a candidate for atypical-speech generalization research and independently designed ASR studies; I do not automatically relabel its unrestricted transcriptions as my co-designer's commands.

Access requests may take days. I will run a schema inspection **only after approval** and then decide how to adapt to the real column names and terms. I will not infer, impersonate or reidentify any speaker. I will not download through an unofficial mirror to evade the gate. When working with restricted data, I record the allowed use and restrictions in the private provenance ledger.

In [ ]:
HF_SOURCES = {
 'cdli_english': 'cdli/kenyan_english_nonstandard_speech_v1.0',
 'cdli_swahili': 'cdli/kenyan_swahili_nonstandard_speech_v1.0',
 'afrispeech': 'intronhealth/afrispeech-200',
 'fleurs': 'google/fleurs',
 'speech_commands': 'google/speech_commands',
}
DATA_APPROVAL = {key:False for key in HF_SOURCES}
PROVENANCE = {key:{'repo_id':v,'access_granted':DATA_APPROVAL[key],
      'reviewed_terms':False,'purpose':'research; no publication of individual audio',
      'split_policy':'preserve publisher partitions'} for key,v in HF_SOURCES.items()}
print(pd.DataFrame.from_dict(PROVENANCE,orient='index')[['repo_id','access_granted','reviewed_terms']])

In [ ]:
INSPECT_GATED_CORPORA=False   # I only set true after access and explicit terms review.
def inspect_dataset_config(repo_id, accepted_terms=False):
    if not accepted_terms:raise PermissionError('I must read and accept the publisher’s terms first.')
    from datasets import get_dataset_config_names,get_dataset_split_names,load_dataset
    # authenticate using the locally cached `hf auth login` credential; never paste tokens into a notebook.
    configs=get_dataset_config_names(repo_id,token=True)
    print('Available configs:',configs)
    config=configs[0] if configs else None
    splits=get_dataset_split_names(repo_id,name=config,token=True)
    print('Published splits:',splits)
    # One streamed row is enough for schema discovery; this still contacts and reads the gated dataset.
    sample=next(iter(load_dataset(repo_id,name=config,split=splits[0],streaming=True,token=True)))
    print('Column names:',list(sample.keys()))
    print('Value types only:',{k:type(v).__name__ for k,v in sample.items()})
    return {'config':config,'splits':splits,'columns':list(sample)}
if INSPECT_GATED_CORPORA:
    for ds in ('cdli_english','cdli_swahili'):
        if not DATA_APPROVAL[ds]:raise PermissionError(f'Approval not recorded for {ds}')
        schema=inspect_dataset_config(HF_SOURCES[ds],accepted_terms=True)
        print(ds,schema)
else:print('Gated dataset inspection not run; no data access assumed.')

### 1B. Schema harmonization and provenance

I require an explicit mapping from the publisher's actual fields to our private manifest rather than silently guessing `speaker_id`, `transcript`, `audio`, `session_id` or license scope. Public training data and a consenting co-designer are different populations and need separate research tables. I preserve a data source identifier, revision, audio hash, original split and allowed-use flag. For reproducible analysis, I record dataset counts and distributions, but avoid publishing protected samples or speaker attributes.

**Dataset specific evaluation:** For CDLI I can measure ASR WER against its official test partition **only if that partition has not already been used to select, train or tune the evaluated model**; since the published CDLI checkpoints were trained on that corpus's training partition, published CDLI test WER is reference context, not new independent evidence. For command personalization I use fresh, consented enrollment and independent test sessions.

In [ ]:
def file_sha256(path, block=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(block)
            if not b:break
            h.update(b)
    return h.hexdigest()

def audit_source_manifest(df,expected_source):
    if 'source' not in df or 'split' not in df:raise ValueError('Source and split required')
    if len(df) and not df.source.eq(expected_source).all():raise ValueError('Unexpected source mixed into cohort')
    if len(df) and not df.split.isin(['train','val','test','external_test']).all():
        raise ValueError('Invalid split')
    return {'rows':len(df),'source':expected_source,'by_split':df.split.value_counts().to_dict()}
print('Data lineage helpers ready.')

## 2. Environment and deterministic experiment configuration

I use Python 3.11 or 3.12 in a clean virtual environment. For the official toolkit I separately use Node.js 20 or later. **Heavy packages are deliberately not downloaded or installed automatically**, because training machines may be offline or have limited VRAM. Install after inspecting the optional commands.

Example environment commands, run in a terminal:

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# macOS/Linux: source .venv/bin/activate
python -m pip install -U pip
python -m pip install numpy pandas scipy scikit-learn soundfile librosa matplotlib seaborn tqdm joblib pydantic fastapi uvicorn python-multipart huggingface_hub
# Select the correct PyTorch CUDA/CPU wheel from https://pytorch.org/get-started/locally/
python -m pip install torch torchaudio
python -m pip install 'transformers>=4.44,<5' 'datasets>=2.20,<5' accelerate evaluate jiwer safetensors
node --version
# Clone toolkit in a separate terminal at a pinned commit on a machine with Internet.
git clone https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit.git toolkit-upstream
```

I record the exact versions and toolkit commit after installation; the version ranges above are installation guidance, not a verified lockfile. In a serious run I freeze dependencies and record model revisions and content hashes. I avoid Python 3.13 for this project unless all audio/PyTorch dependencies are supported.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, random, time, platform, math, re
import numpy as np
import pandas as pd
SEED = 20261102
random.seed(SEED); np.random.seed(SEED)
ROOT = Path.cwd() / "team4_voice_lab"
PRIVATE = ROOT / "private"     # NEVER commit this directory
PUBLIC = ROOT / "public_results"
ARTIFACTS = PRIVATE / "models"
for folder in (PRIVATE, PUBLIC, ARTIFACTS): folder.mkdir(parents=True, exist_ok=True)
CONFIG = {
    "seed": SEED, "sample_rate": 16000, "min_sec": 0.25, "max_sec": 10.0,
    "encoder_id": "facebook/wav2vec2-xls-r-300m",
    "alternate_encoder_id": "facebook/wav2vec2-base",
    "english_asr_id": "cdli/whisper-small_finetuned_kenyan_english_nonstandard_speech_v1.0",
    "swahili_asr_id": "cdli/whisper-small_finetuned_kenyan_swahili_nonstandard_speech_v1.0",
    "general_asr_id": "openai/whisper-small",
    "device": "cuda", "batch_size": 4, "feature_batch_size": 4,
    "max_false_accept_rate_validation": 0.02, "min_negative_validation": 20,
    "model_revision": None, "training_enabled": False,
}
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
print({"python":sys.version.split()[0],"platform":platform.platform(),"root":str(ROOT),"run_id":RUN_ID})

### Data governance and consent gates

For personal recordings, I require explicit and informed consent for a specific purpose, an agreed retention/deletion policy, a private storage location and an option to withdraw. I never commit audio, raw transcripts that identify a person, speaker identifiers linked to identity, biometric embeddings or speaker-specific model weights. I do not infer medical diagnoses, severity, gender, age or identity from audio. I record whether cloud processing is authorized separately; local inference is the default preference. The public repository receives only source code, approved deidentified aggregate metrics and methodology.

**Minimum dataset manifest contract:** `audio_path`, `speaker_id` (pseudonym), `session_id`, `language` (`en-KE` or `sw-KE`), `command_id`, `consent_training`, `consent_evaluation`, `source`, `allowed_use`, `split`, `recorded_at`. `command_id` is a **semantic intent**, not a guessed ASR transcription. Negative examples get `UNKNOWN`. A person's own words may differ from our canonical utterance.

**Kenya-specific legal checkpoint:** The Kenyan Data Protection Act, 2019 explicitly includes voice recognition in its biometric-data definition; the Office of the Data Protection Commissioner describes audio and biometric information as personal data and provides a specific biometric-data guidance note. I obtain appropriately informed opt-in consent and review the organizer's lawful handling, storage, deletion and sharing obligations rather than assuming a hackathon exemption. This is not legal advice. Sources: [Kenya Law, Data Protection Act 2019](https://kenyalaw.org/kl/fileadmin/pdfdownloads/Acts/2019/TheDataProtectionAct__No24of2019.pdf), [ODPC FAQ](https://www.odpc.go.ke/faqs/), [ODPC guidance on biometric data](https://www.odpc.go.ke/wp-content/uploads/2025/11/ODPC-%E2%80%93-Guidance-Note-on-Biometric-Data.pdf).


In [ ]:
from dataclasses import dataclass
from typing import Optional
COMMANDS = {
    "SCROLL_DOWN": {"canonical": "scroll down", "risk": "low", "languages": ["en-KE", "sw-KE"]},
    "SCROLL_UP": {"canonical": "scroll up", "risk": "low", "languages": ["en-KE", "sw-KE"]},
    "GO_BACK": {"canonical": "go back", "risk": "low", "languages": ["en-KE", "sw-KE"]},
    "READ_PAGE": {"canonical": "read this", "risk": "low", "languages": ["en-KE", "sw-KE"]},
    "UNDO": {"canonical": "undo", "risk": "low", "languages": ["en-KE", "sw-KE"]},
}
# I will replace these provisional commands after co-design. The canonical mapping
# deliberately targets phrases the verified toolkit grammar already understands.
NEGATIVE = "UNKNOWN"
COLUMNS = ["audio_path","speaker_id","session_id","language","command_id",
           "consent_training","consent_evaluation","source","allowed_use","split","recorded_at"]
MANIFEST_PATH = PRIVATE / "co_designer_manifest.csv"
if not MANIFEST_PATH.exists():
    pd.DataFrame(columns=COLUMNS).to_csv(MANIFEST_PATH,index=False)
print("Private blank manifest:",MANIFEST_PATH)
print("Provisional intents:",COMMANDS)

## 3. Enrollment protocol and audio collection

I first ask the co-designer what they want to do, their preferred language and phrases, their preferred recording initiation method, and whether a nonvoice option is preferable. I do not prescribe the pronunciation of a command. I test at least one meaningful complete task (for example, navigate and read a page). All recordings are voluntary, short, pausable and locally managed. I record test sessions separately from enrollment whenever feasible.

**Suggested progressive pilot only, not an evidence-backed minimum:** 5 selected intentions, initially 5 to 10 enrollment examples per intention, plus an appropriate number of distinct background/other-speech negatives; pause and reassess fatigue. Validate in a different session. Do not mix a later test session into iterative tuning. If the participant prefers fewer recordings, reduce scope rather than pressure them.

I encourage team members to use a trustworthy recording tool and export 16 bit PCM WAV; notebook accepts other audio formats supported by SoundFile, but refuses missing/corrupt/implausible files. Do not use the browser's automatically transcribed text as the ground-truth label for atypical speech.

In [ ]:
def validate_manifest(frame: pd.DataFrame, require_files=True) -> pd.DataFrame:
    missing = set(COLUMNS) - set(frame.columns)
    if missing: raise ValueError(f"Missing manifest columns: {sorted(missing)}")
    frame = frame.copy()
    if not len(frame):
        print("Manifest empty: I must collect approved recordings before model training.")
        return frame
    for col in ["audio_path","speaker_id","session_id","language","command_id","source","allowed_use","split"]:
        if frame[col].isna().any() or frame[col].astype(str).str.strip().eq("").any():
            raise ValueError(f"Null/blank values in {col}")
    if not frame.language.isin(["en-KE","sw-KE"]).all(): raise ValueError("Unsupported language labels")
    if not frame.command_id.isin([*COMMANDS,NEGATIVE]).all(): raise ValueError("Unknown command IDs")
    if not frame.split.isin(["train","val","test","external_test"]).all(): raise ValueError("Unrecognized split")
    for flag in ["consent_training","consent_evaluation"]:
        frame[flag] = frame[flag].astype(str).str.lower().map({"true":True,"false":False,"1":True,"0":False})
        if frame[flag].isna().any():raise ValueError(f"Invalid consent values in {flag}")
    if (frame.loc[frame.split=="train","consent_training"]==False).any():
        raise PermissionError("Training split contains recordings without training consent")
    if (frame.loc[frame.split.isin(["val","test","external_test"]),"consent_evaluation"]==False).any():
        raise PermissionError("Evaluation split contains recordings without evaluation consent")
    paths = [Path(v).expanduser().resolve() for v in frame.audio_path]
    if len(paths)!=len(set(paths)): raise ValueError("Same audio file appears more than once; data leakage")
    if require_files and any(not pp.is_file() for pp in paths):
        raise FileNotFoundError("Manifest references missing audio file(s)")
    frame["audio_path"] = [str(pp) for pp in paths]
    return frame
manifest = validate_manifest(pd.read_csv(MANIFEST_PATH))
print("Rows available:",len(manifest))

### Speaker and session leakage tests

For public CDLI experiments I **preserve the publisher's speaker/phrase-disjoint train, dev, test partitions**, never joining their test speakers into training. For personal enrollment, the SAME consenting individual must appear in enrollment and eventual evaluation by design, but the **recording sessions for model fitting, threshold tuning and final testing must be distinct**. Both are valid experiments, but they answer different questions and must never be combined into a single published metric. Public corpora cannot substitute for the prospective person's independent test set.

In [ ]:
def assert_split_integrity(df: pd.DataFrame, personal=True):
    if df.empty: return {"status":"no-data"}
    # File checks and session independence are separate, mandatory assertions.
    for sp in ["train","val","test"]:
        sub=df[df.split.eq(sp)]
        if sub.empty: print(f"WARNING: {sp} has no data")
    if personal:
        per_split = {sp:set(df.loc[df.split.eq(sp),"session_id"].tolist()) for sp in ["train","val","test"]}
        for a,b in [("train","val"),("train","test"),("val","test")]:
            if per_split[a]&per_split[b]:raise AssertionError(f"Recording sessions overlap: {a}/{b}")
    else:
        tr=set(df.loc[df.split.eq('train'),"speaker_id"])
        te=set(df.loc[df.split.eq('test'),"speaker_id"])
        if tr&te: raise AssertionError("Public speaker-disjoint benchmark violated")
    return {"status":"pass","counts":df.split.value_counts().to_dict()}
print(assert_split_integrity(manifest))

## 4. Audio quality control and exploratory analysis

I standardize mono audio and resample with a proper anti-aliasing filter instead of relabeling a recording's sample rate. I flag clipping, silence, ultra short speech and unusual duration for manual review; I do **not** delete atypical speech merely because it has low energy. I do not use aggressive denoising or speed/pitch augmentation until the co-designer verifies that it does not distort meaningful individual characteristics.

In [ ]:
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd

def read_audio(path, target_sr=16000):
    audio, sr = sf.read(str(path), always_2d=True, dtype="float32")
    if sr <= 0 or audio.size==0: raise ValueError("Invalid audio")
    # Multi-channel average, not a lossy reshape.
    x=audio.mean(axis=1)
    if sr != target_sr:
        g=gcd(int(sr),int(target_sr)); x=resample_poly(x,target_sr//g,sr//g).astype(np.float32)
    if not np.isfinite(x).all():raise ValueError("NaN or Inf in recording")
    return np.asarray(x,dtype=np.float32),target_sr

def audio_qa(path):
    x,sr=read_audio(path)
    duration=len(x)/sr
    return {"duration_s":duration,"peak":float(np.max(np.abs(x))),
            "rms":float(np.sqrt(np.mean(x*x))),
            "clip_fraction":float(np.mean(np.abs(x)>=0.999)),
            "is_silent":bool(np.sqrt(np.mean(x*x))<0.002),
            "outside_duration":bool(not CONFIG['min_sec']<=duration<=CONFIG['max_sec'])}

if not manifest.empty:
    qa = pd.DataFrame([audio_qa(p) for p in manifest.audio_path])
    report = pd.concat([manifest.reset_index(drop=True)[['language','command_id','split','source']],qa],axis=1)
    print(report.groupby(['language','split']).agg(n=('duration_s','size'),duration_min=('duration_s',lambda v:round(v.sum()/60,2)),clipped=('clip_fraction',lambda v:int((v>0.01).sum()))))
    print('FLAGGED for human review:',int((report.outside_duration|report.is_silent|(report.clip_fraction>.01)).sum()))
else:
    print("QA awaits consented recordings.")

## 5. Reproducibility controls, model caching and compute budget

I will cache pretrained weights locally, record the exact model revision and run a device diagnostic before experiments. XLS-R 300M may require substantial GPU memory, especially for end-to-end fine tuning; the frozen-embedding path is my baseline because it is cheaper and auditable. I do not upload private audio to any cloud API without explicit permission. If CPU-only, I use a smaller model and document the deviation, not secretly change the research question.

In [ ]:
import importlib.util, importlib.metadata
PKGS=['torch','torchaudio','transformers','datasets','scikit-learn','soundfile','scipy','numpy','pandas','fastapi']
versions={}
for package in PKGS:
    try:versions[package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:versions[package]='NOT INSTALLED'
print(pd.Series(versions))
try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Compute device:',DEVICE, 'CUDA:',torch.version.cuda)
    torch.manual_seed(SEED)
except ImportError:
    DEVICE='cpu'
    print('Torch not installed; ML cells are optional until dependencies are installed.')
metadata={'date_utc':pd.Timestamp.now(tz='UTC').isoformat(),'seed':SEED,'config':CONFIG,'packages':versions,'compute_device':DEVICE}
(PUBLIC/'environment_manifest.json').write_text(json.dumps(metadata,indent=2,default=str))

## 6. Ordinary ASR and Kenya-adapted ASR baselines

I compare a generic model to the two CDLI Kenyan atypical speech models with **matching per-language decoding protocols**. For English use `language='english'`; for Kiswahili use `language='swahili'` when supported by the processor. I only map transcripts to command intents through an explicit, user-approved table; I never guess that a semantically similar sentence authorizes an action. Bilingual code switching may defeat a forced language decoder; I report that separately and test an automatic-language mode as an ablation.

**Important:** CDLI English and Swahili models were already adapted on CDLI data. Their published metrics cannot be used as an independent evaluation on those same test sets if I reuse them for training or make new thresholds using the official test set.

In [ ]:
import unicodedata
# These are examples, not claims about what the participant actually says.
PHRASES = {
  'en-KE': {'scroll down':'SCROLL_DOWN','down':'SCROLL_DOWN','scroll up':'SCROLL_UP','up':'SCROLL_UP',
            'go back':'GO_BACK','back':'GO_BACK','read this':'READ_PAGE','read page':'READ_PAGE',
            'undo':'UNDO'},
  'sw-KE': {'shuka chini':'SCROLL_DOWN','sogeza chini':'SCROLL_DOWN',
            'panda juu':'SCROLL_UP','sogeza juu':'SCROLL_UP',
            'rudi nyuma':'GO_BACK','nirudishe nyuma':'GO_BACK',
            'soma ukurasa':'READ_PAGE','nisomee':'READ_PAGE','tengua':'UNDO'}
}
def normalize_text(s):
    s=unicodedata.normalize('NFKC',str(s or '')).lower()
    s=re.sub(r'[^\w\s]', ' ',s,flags=re.UNICODE)
    return ' '.join(s.split())
def map_transcript_to_command(text, language):
    return PHRASES.get(language,{}).get(normalize_text(text),NEGATIVE)

# Verify refusals. This strict mapper cannot execute arbitrary free-form prompts.
assert map_transcript_to_command('Sogeza chini!','sw-KE')=='SCROLL_DOWN'
assert map_transcript_to_command('Hi, how are you?', 'en-KE')==NEGATIVE
assert map_transcript_to_command('please transfer money', 'en-KE')==NEGATIVE
print('Exact-phrase mapping checks passed; customize only with the co-designer.')

In [ ]:
def load_asr_pipeline(model_id, language):
    """Optional heavy download. Use only if weights may be downloaded under model terms."""
    from transformers import pipeline
    import torch
    device=0 if torch.cuda.is_available() else -1
    return pipeline('automatic-speech-recognition',model=model_id,device=device)

def transcribe_one(pipe, audio_path, language=None):
    x,sr=read_audio(audio_path)
    kw={'task':'transcribe'}
    if language:kw['language']=language
    start=time.perf_counter()
    result=pipe({'array':x,'sampling_rate':sr},generate_kwargs=kw, return_timestamps=False)
    return {'transcript':result['text'].strip(),'latency_s':time.perf_counter()-start}

RUN_ASR=False  # Explicitly enable after approved data, dependencies and compute exist
if RUN_ASR:
    assert not manifest.empty, 'I need real approved audio, not empty demo data.'
    asr_rows=[]
    for lang,mid,whisper_lang in [('en-KE',CONFIG['english_asr_id'],'english'),('sw-KE',CONFIG['swahili_asr_id'],'swahili')]:
        pipe=load_asr_pipeline(mid,whisper_lang)
        for _,row in manifest.loc[(manifest.language==lang)&(manifest.split.isin(['val','test']))].iterrows():
            pred=transcribe_one(pipe,row.audio_path,whisper_lang)
            asr_rows.append({'audio_path':row.audio_path,'split':row.split,'true':row.command_id,
                'language':lang,'model':mid,**pred,'pred':map_transcript_to_command(pred['transcript'],lang)})
        del pipe
    asr_predictions=pd.DataFrame(asr_rows)
    asr_predictions.to_csv(PRIVATE/'asr_predictions_PRIVATE.csv',index=False)
    print(asr_predictions.groupby(['language','split','model']).apply(lambda g:(g.true==g.pred).mean()))
else:
    print('ASR not run. No baseline scores claimed.')

### 6A. A fair four-model transcription comparison

I compare **ordinary Whisper Small** and **CDLI Kenyan English Whisper Small** on the *same English test recordings*. For Kiswahili I compare **the published Swahili base `cdli/whisper-small-Swahili_finetuned_small_CV20`** with the corresponding **CDLI Kenyan Swahili adapted checkpoint** on the same Kiswahili test recordings. I do **not** call `openai/whisper-small` the publisher's Swahili baseline because the CDLI model card uses a different reference checkpoint.

If the recorded phrases are in both languages within one utterance, I additionally test language detection instead of forcing a single language. **No test data enters the phrase table or tuning process**. I compute **WER only when the human reference transcript exists** and **task command accuracy / false activation only when a co-designer-approved intent label exists**.

In [ ]:
ASR_COMPARISONS={
 'en-KE':[
   ('ordinary_whisper_small','openai/whisper-small','english'),
   ('cdli_kenyan_english',CONFIG['english_asr_id'],'english')],
 'sw-KE':[
   ('swahili_whisper_cv20','cdli/whisper-small-Swahili_finetuned_small_CV20','swahili'),
   ('cdli_kenyan_swahili',CONFIG['swahili_asr_id'],'swahili')],
}

def run_paired_asr_benchmark(df,language,cap=None):
    from jiwer import wer
    subset=df[(df.language==language)&(df.split=='test')].copy()
    if subset.empty:raise ValueError('Need held-out test audio')
    # A cap is only allowed for a labeled pilot, not a claim of full-benchmark comparison.
    if cap is not None:subset=subset.head(cap)
    outputs=[]
    for name,model_id,decode_language in ASR_COMPARISONS[language]:
        pipe=load_asr_pipeline(model_id,decode_language)
        for _,row in subset.iterrows():
            result=transcribe_one(pipe,row.audio_path,decode_language)
            outputs.append({ 'model':name,'model_id':model_id,'language':language,
                'record_key':hashlib.sha256(row.audio_path.encode()).hexdigest()[:12],
                'gold_command':row.command_id,'pred_command':map_transcript_to_command(result['transcript'],language),
                'transcript':result['transcript'],'latency_s':result['latency_s'],
                'gold_transcript':row.get('verified_transcript',np.nan)})
        del pipe
    out=pd.DataFrame(outputs)
    summaries=[]
    for model_name,g in out.groupby('model'):
        summary={'model':model_name,'n':len(g),
          'intent_accuracy':float((g.gold_command==g.pred_command).mean()),
          'false_accepts':int(((g.gold_command==NEGATIVE)&(g.pred_command!=NEGATIVE)).sum()),
          'latency_median_s':float(g.latency_s.median())}
        valid=g.gold_transcript.notna()
        if valid.any():
            summary['wer']=float(wer(g.loc[valid,'gold_transcript'].map(normalize_text).tolist(),
                                     g.loc[valid,'transcript'].map(normalize_text).tolist()))
        summaries.append(summary)
    return pd.DataFrame(summaries),out

RUN_PAIRED_ASR=False
if RUN_PAIRED_ASR:
    assert not manifest.empty
    for language in ['en-KE','sw-KE']:
        if manifest[(manifest.language==language)&(manifest.split=='test')].empty:
            print('No test audio for',language);continue
        summary,private_preds=run_paired_asr_benchmark(manifest,language)
        private_preds.to_csv(PRIVATE/f'paired_asr_{language}_PRIVATE.csv',index=False)
        print(summary)
else:print('Fair paired ASR experiment defined, not executed.')

## 7. Pretrained XLS-R embeddings

My model selection is language informed. XLS-R was pretrained on speech from 128 languages, while Wav2Vec2 Base is a lighter English-centric comparator. **No general model guarantees transfer to this speaker.** I extract hidden representations using masks so padding is excluded, L2 normalize the resulting vectors, and cache them **only inside private storage**. The encoder outputs are not anonymized; they may retain speaker-related information.

I use CPU if that is all that is available. If no encoder can be downloaded, I can still run the contract tests at the end, but cannot claim model quality.

In [ ]:
def initialize_encoder(model_id=None):
    """Download weights only when I explicitly run this cell with the dependencies installed."""
    import torch
    from transformers import AutoFeatureExtractor, AutoModel
    model_id=model_id or CONFIG['encoder_id']
    extractor=AutoFeatureExtractor.from_pretrained(model_id,revision=CONFIG['model_revision'])
    model=AutoModel.from_pretrained(model_id,revision=CONFIG['model_revision']).to(DEVICE)
    model.eval()
    return extractor,model

def embed_audio_batch(paths, extractor, model, batch_size=4):
    import torch
    result=[]
    for start in range(0,len(paths),batch_size):
        waveforms=[read_audio(p)[0] for p in paths[start:start+batch_size]]
        padded=extractor(waveforms,sampling_rate=16000,padding=True,return_attention_mask=True,return_tensors='pt')
        tensor_in={k:v.to(DEVICE) for k,v in padded.items()}
        with torch.inference_mode():
            output=model(**tensor_in)
            h=output.last_hidden_state
            # Transformer frames are shorter than original waveform sample count.
            if hasattr(model,'_get_feature_vector_attention_mask') and 'attention_mask' in tensor_in:
                mask=model._get_feature_vector_attention_mask(h.shape[1],tensor_in['attention_mask'])
                lengths=mask.sum(1,keepdim=True).clamp(min=1)
                vec=(h*mask.unsqueeze(-1)).sum(1)/lengths
            else:
                raise RuntimeError('Mask-aware pooling unsupported by this checkpoint; inspect model outputs')
            vec=torch.nn.functional.normalize(vec,p=2,dim=-1)
            result.append(vec.float().cpu().numpy())
    return np.concatenate(result,axis=0)

RUN_EMBEDDINGS=False
if RUN_EMBEDDINGS:
    assert len(manifest)>0,'Approved recordings required'
    extractor,encoder=initialize_encoder()
    X=embed_audio_batch(manifest.audio_path.tolist(),extractor,encoder,CONFIG['feature_batch_size'])
    np.savez_compressed(PRIVATE/'embeddings_PRIVATE.npz',X=X)
    print('Private embeddings:',X.shape)
else:
    print('Embedding model not downloaded or run.')

## 8. Enrollment strategies: prototype similarity and logistic regression

I compare two personalized alternatives over the **same** frozen audio embeddings:

**A. Prototypical enrollment:** compute one centroid per command from the training session and assign only if similarity clears a validation-derived threshold. This can be updated without gradient training.

**B. Discriminative classification:** fit a regularized, class-balanced logistic regression on enrollment embeddings including unknown speech. This requires sufficient class diversity and an independent validation session to tune the rejection rule. I avoid treating the classifier's raw max softmax probability as calibrated confidence.

**Leakage prevention:** The same co-designer may appear in train and test (personalization), but no recordings from the final test session may be used in model selection or threshold calibration.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score

class PrototypeRecognizer:
    def fit(self,X,y):
        classes=sorted(set(y)-{NEGATIVE})
        if not classes:raise ValueError('Need positive commands')
        self.classes_=np.array(classes)
        self.centers_=np.vstack([X[np.array(y)==c].mean(axis=0) for c in classes])
        norm=np.linalg.norm(self.centers_,axis=1,keepdims=True).clip(1e-12)
        self.centers_/=norm
        return self
    def predict_scores(self,X):
        A=np.asarray(X,float)
        A=A/np.linalg.norm(A,axis=1,keepdims=True).clip(1e-12)
        return A@self.centers_.T

class LogisticRecognizer:
    def fit(self,X,y):
        self.clf=LogisticRegression(C=0.5,max_iter=2000,class_weight='balanced',random_state=SEED)
        self.clf.fit(X,y)
        self.classes_=self.clf.classes_
        return self
    def predict_scores(self,X):return self.clf.predict_proba(X)

# Reject UNKNOWN from candidate action classes; unknown remains training negative for logistic regression.
def scores_and_classes(model,X):
    scores=model.predict_scores(X)
    classes=np.array(model.classes_)
    keep=classes!=NEGATIVE
    if not keep.any():raise ValueError('No executable classes')
    return scores[:,keep],classes[keep]

## 9. Threshold calibration, selective risk and an unknown action

A zero-action rejection option is mandatory. I select a single global acceptance threshold on **validation only**, searching over candidate score thresholds. I report false acceptances on negative utterances and accepted-command recall. With limited negative trials, a numerical false activation target cannot be certified; I report the confidence interval and the number of trials.

**A subtle but critical point:** high softmax probability or high cosine similarity by itself does not prove correctness. If a negative example is very close to a command prototype, the model must be evaluated for real false activation risk. I prefer a conservative reject policy to speculative execution. The person has an explicit retry, nonvoice input and physical stop/cancel route.

In [ ]:
from scipy.stats import beta

def predict_with_rejection(scores,classes,threshold):
    top=np.argmax(scores,axis=1)
    selected=classes[top].astype(object)
    selected[np.max(scores,axis=1)<threshold]=NEGATIVE
    return selected

def threshold_search(scores,classes,truth,max_far=.02,min_neg=20):
    truth=np.asarray(truth)
    if np.sum(truth==NEGATIVE)<min_neg:
        raise ValueError(f'Insufficient independent negatives for threshold calibration: require >= {min_neg}')
    # Calibration ONLY on validation; test set is never consulted here.
    observed=np.sort(np.unique(scores.max(axis=1)))
    candidates=np.r_[-np.inf,observed,np.nextafter(observed.max(),np.inf)]
    table=[]
    for th in candidates:
        pred=predict_with_rejection(scores,classes,th)
        unknown=truth==NEGATIVE
        pos=~unknown
        fa=int(np.sum((pred!=NEGATIVE)&unknown))
        far=fa/max(1,int(unknown.sum()))
        recall=float(np.mean(pred[pos]==truth[pos])) if pos.any() else 0.0
        table.append((float(th),far,recall,fa))
    feasible=[row for row in table if row[1]<=max_far]
    if not feasible:raise RuntimeError('No validation-safe threshold; do not activate voice actions')
    # Maximize valid-command recall among thresholds meeting the FAR constraint.
    chosen=sorted(feasible,key=lambda row:(-row[2],row[1],row[0]))[0]
    return {'threshold':chosen[0],'val_far':chosen[1],'val_accepted_correct_recall':chosen[2],
            'val_false_activations':chosen[3],'validation_negatives':int((truth==NEGATIVE).sum())}

def clopper_pearson(k,n,alpha=.05):
    if n<=0:return (float('nan'),float('nan'))
    lo=0. if k==0 else beta.ppf(alpha/2,k,n-k+1)
    hi=1. if k==n else beta.ppf(1-alpha/2,k+1,n-k)
    return float(lo),float(hi)

# Guardrail unit tests without any speech model weights.
assert predict_with_rejection(np.array([[.1,.2],[.95,.1]]),np.array(['BACK','READ']),.8).tolist()==['UNKNOWN','BACK']
assert clopper_pearson(0,100)[1]>0.02 # zero errors does not prove a 2% upper FAR
print('Safety and uncertainty unit tests passed.')

## 10. End to end frozen-encoder experiment (real audio required)

This cell jointly fits prototypes and a linear classifier. It uses fixed splits, selects thresholds on validation **without peeking at test**, and evaluates only after selecting a candidate. It fails loudly if the manifest is incomplete or missing negatives, classes, or sessions. If training recordings are not yet available, it does not fabricate a leaderboard.

In [ ]:
def require_experiment_data(df):
    if df.empty:raise RuntimeError('I have not enrolled the co-designer or obtained approved audio.')
    assert_split_integrity(df,personal=True)
    for sp in ['train','val','test']:
        if not len(df.loc[df.split==sp]):raise ValueError(f'Missing {sp} split')
    if not set(COMMANDS).issubset(set(df.loc[df.split=='train','command_id'])):
        raise ValueError('Train recordings do not cover all active commands')
    if (df.loc[df.split=='val','command_id']==NEGATIVE).sum()<CONFIG['min_negative_validation']:
        raise ValueError('Need independent unknown-speech validation trials')
    if not df.loc[df.split=='test'].command_id.eq(NEGATIVE).any():
        raise ValueError('Unknown speech absent from final test')

def evaluate_predictions(truth,pred):
    truth=np.array(truth);pred=np.array(pred)
    negatives=truth==NEGATIVE
    positives=~negatives
    fa=int(np.sum(negatives&(pred!=NEGATIVE)))
    rejected_correct=int(np.sum(positives&(pred==NEGATIVE)))
    return {'n':len(truth),'macro_f1_including_unknown':f1_score(truth,pred,average='macro',zero_division=0),
        'overall_accuracy':accuracy_score(truth,pred),
        'accepted_correct_recall':float(np.mean(pred[positives]==truth[positives])) if positives.any() else np.nan,
        'false_accept_count':fa,'unknown_n':int(negatives.sum()),
        'false_accept_rate':fa/max(1,int(negatives.sum())),
        'false_accept_ci95':clopper_pearson(fa,int(negatives.sum())),
        'rejected_valid_count':rejected_correct}

RUN_FROZEN_EXPERIMENT=False
if RUN_FROZEN_EXPERIMENT:
    require_experiment_data(manifest)
    if 'X' not in globals():raise RuntimeError('Run the embedding extraction cell first')
    idx={sp:np.flatnonzero(manifest.split.to_numpy()==sp) for sp in ['train','val','test']}
    XX={sp:X[ii] for sp,ii in idx.items()}
    yy={sp:manifest.command_id.to_numpy()[ii] for sp,ii in idx.items()}
    candidates={'prototype':PrototypeRecognizer().fit(XX['train'],yy['train']),
                'logistic':LogisticRecognizer().fit(XX['train'],yy['train'])}
    fitted={}
    for name,model in candidates.items():
        val_scores,classes=scores_and_classes(model,XX['val'])
        policy=threshold_search(val_scores,classes,yy['val'],max_far=CONFIG['max_false_accept_rate_validation'],min_neg=CONFIG['min_negative_validation'])
        fitted[name]={'model':model,'classes':classes,'policy':policy}
        print('VALIDATION:',name,policy)
    # Freeze selection based on validation before opening the final test.
    best_name=max(fitted,key=lambda k:(fitted[k]['policy']['val_accepted_correct_recall'],-fitted[k]['policy']['val_far']))
    chosen=fitted[best_name]
    test_scores,test_classes=scores_and_classes(chosen['model'],XX['test'])
    test_pred=predict_with_rejection(test_scores,test_classes,chosen['policy']['threshold'])
    final_report=evaluate_predictions(yy['test'],test_pred)
    print('CHOSEN',best_name,'HELD OUT TEST',final_report)
else:
    print('No real experiment run; I will not invent performance values.')

## 11. Selective encoder fine tuning: a controlled secondary experiment

I do **not** start by fine tuning 300 million parameters on 40 recordings. After the frozen models are evaluated on validation, I may run a *parameter-limited* experiment with the top encoder layer(s) and a new classification head. I freeze feature extraction and most transformer blocks, optimize cross entropy with class weights, gradient clipping and early stopping on the **validation loss**, then reselect a rejection threshold on validation. Final test remains sealed until all choices are frozen.

This section is intentionally opt in because a normal laptop may be unable to train XLS-R 300M. I report hardware, training time and peak memory; if there is too little data, I decline to fine tune rather than fabricate confidence. No code here uses pretrained ASR transcript pseudo labels as ground truth for the co-designer.

In [ ]:
# These classes are defined lazily so the notebook remains inspectable without Transformers.
def build_partial_finetune_model(model_id, n_labels, train_last_n=1):
    import torch
    from transformers import Wav2Vec2ForSequenceClassification, AutoFeatureExtractor
    feature_extractor=AutoFeatureExtractor.from_pretrained(model_id)
    model=Wav2Vec2ForSequenceClassification.from_pretrained(
        model_id,num_labels=n_labels,ignore_mismatched_sizes=True)
    for parameter in model.parameters():parameter.requires_grad=False
    # The HF classification head can be named classifier/projector/projector; unfreeze only these.
    for key in ('projector','classifier'):
        if hasattr(model,key):
            for parameter in getattr(model,key).parameters():parameter.requires_grad=True
    if hasattr(model,'wav2vec2') and train_last_n>0:
        layers=model.wav2vec2.encoder.layers
        for layer in layers[-train_last_n:]:
            for parameter in layer.parameters():parameter.requires_grad=True
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    total=sum(p.numel() for p in model.parameters())
    assert 0<trainable<total,'Partial fine tuning misconfigured'
    print('Trainable parameters:',trainable,'/',total,'ratio:',round(trainable/total,5))
    return model,feature_extractor

def collate_waveform_batch(paths,labels,feature_extractor,device):
    import torch
    xs=[read_audio(x)[0] for x in paths]
    inputs=feature_extractor(xs,padding=True,sampling_rate=16000,return_attention_mask=True,return_tensors='pt')
    return {k:v.to(device) for k,v in inputs.items()},torch.tensor(labels,dtype=torch.long,device=device)

def fine_tune_partial(df,model_id,epochs=8,lr=2e-5,last_n=1):
    import torch
    from sklearn.utils.class_weight import compute_class_weight
    from torch.optim import AdamW
    from torch.utils.data import DataLoader
    require_experiment_data(df)
    if DEVICE!='cuda':raise RuntimeError('This research training path requires an available GPU; use frozen encoder on CPU')
    labels=sorted(df.loc[df.split=='train','command_id'].unique())
    if NEGATIVE not in labels:raise ValueError('Training requires UNKNOWN negatives')
    y_to_id={v:i for i,v in enumerate(labels)}
    model,feature_extractor=build_partial_finetune_model(model_id,len(labels),last_n)
    model.to(DEVICE)
    tr=df[df.split=='train'];val=df[df.split=='val']
    class_values=compute_class_weight('balanced',classes=np.array(labels),y=tr.command_id.to_numpy())
    criterion=torch.nn.CrossEntropyLoss(weight=torch.tensor(class_values,dtype=torch.float32,device=DEVICE))
    optimizer=AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=.01)
    def batches(frame,shuffle):
        ids=np.arange(len(frame))
        if shuffle:np.random.default_rng(SEED).shuffle(ids)
        for chunk in np.array_split(ids,max(1,int(np.ceil(len(ids)/2)))):
            yield collate_waveform_batch(frame.iloc[chunk].audio_path.tolist(),
                  [y_to_id[v] for v in frame.iloc[chunk].command_id],feature_extractor,DEVICE)
    best=float('inf');best_weights=None;patience=3;stale=0;history=[]
    for epoch in range(epochs):
        model.train();train_losses=[]
        for inputs,y in batches(tr,True):
            optimizer.zero_grad(set_to_none=True)
            output=model(**inputs)
            loss=criterion(output.logits,y)
            loss.backward();torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
            optimizer.step();train_losses.append(float(loss.item()))
        model.eval();validation=[]
        with torch.inference_mode():
            for inputs,y in batches(val,False):
                validation.append(float(criterion(model(**inputs).logits,y).item()))
        loss_val=float(np.mean(validation));history.append({'epoch':epoch+1,'train_loss':float(np.mean(train_losses)),'val_loss':loss_val})
        if loss_val<best-1e-4:
            best=loss_val;stale=0
            best_weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if stale>=patience:break
    model.load_state_dict(best_weights)
    return model,feature_extractor,y_to_id,pd.DataFrame(history)

RUN_PARTIAL_FINETUNE=False
if RUN_PARTIAL_FINETUNE:
    model_ft,extractor_ft,label_to_id,training_history=fine_tune_partial(manifest,CONFIG['encoder_id'])
    print(training_history)
else:
    print('Selective fine tuning is defined but was not performed.')

## 12. Error analysis, per-language reporting and statistical uncertainty

I will report performance by English and Kiswahili separately because overall accuracy can conceal poorer access in one language. I will also separate commands, recording sessions and noise conditions. **Avoid diagnostic or demographic subgroup inference** from audio; use such metadata only when explicitly authorized and statistically appropriate.

For the final co-designer evaluation, one individual is a case study. Confidence intervals over utterances are *not* population-level generalization. I report paired results on the same audio and observe actual independent tasks. I do not calculate p-values over artificially duplicated recordings.

In [ ]:
def stratified_final_report(df, pred):
    if len(df)!=len(pred):raise ValueError('Predictions and ground truth lengths differ')
    report=[]
    for lang,g in df.assign(pred=pred).groupby('language'):
        report.append({'language':lang,**evaluate_predictions(g.command_id,g.pred)})
    return pd.DataFrame(report)

def reproducible_error_table(df, predicted):
    # Private use only: do not publish paths, speaker IDs, or individual voices.
    out=df[['language','session_id','command_id','split']].copy()
    out['prediction']=predicted
    out['error_type']=np.select([
       (out.command_id==NEGATIVE)&(out.prediction!=NEGATIVE),
       (out.command_id!=NEGATIVE)&(out.prediction==NEGATIVE),
       (out.command_id!=NEGATIVE)&(out.prediction!=out.command_id)],
       ['unsafe_false_accept','false_rejection','wrong_command'],default='correct')
    return out

if RUN_FROZEN_EXPERIMENT:
    test_df=manifest.iloc[idx['test']].copy()
    print(stratified_final_report(test_df,test_pred).to_string(index=False))
    private_errors=reproducible_error_table(test_df,test_pred)
    private_errors.to_csv(PRIVATE/'errors_PRIVATE.csv',index=False)
else:
    print('Per-language evaluation requires independently held-out co-designer audio.')

## 13. Data ablations and deployment-relevant tests

I assess robustness to (i) enrollment example count, (ii) language (en-KE/sw-KE), (iii) session shift, (iv) noise, (v) microphone type, (vi) unknown speech, and (vii) ASR vs embedding recognizer. Comparisons must use **the same test examples**. If there is not enough data, I label a comparison not estimable. I avoid destructive augmentations that could invalidate the co-designer's speech patterns. I also measure model loading time, per-utterance inference latency (warm and cold), peak memory and CPU fallback.

**Human factors** are a separate experiment: how did the person start listening, how did they know the microphone was on, did the system provide clear feedback without competing with their assistive technology, could they stop it, and could they complete a chosen task without assistance?

In [ ]:
def measure_inference(fn,items,warmup=2):
    if not items:raise ValueError('Need test items')
    for sample in items[:min(warmup,len(items))]:fn(sample)
    timings=[]
    for sample in items:
        t0=time.perf_counter();fn(sample)
        timings.append(time.perf_counter()-t0)
    return {'n':len(timings),'p50_ms':round(float(np.percentile(timings,50)*1000),1),
            'p95_ms':round(float(np.percentile(timings,95)*1000),1),
            'max_ms':round(float(max(timings)*1000),1)}
print('Inference benchmark defined; report only when running real model on real test audio.')

## 14. Decision freeze and model artifact export

The exported recognizer must carry the exact encoder checkpoint, label order, decision threshold, feature normalization and training provenance. The encrypted or private personalized model **must not be pushed to the public toolkit repository**. The public repository may contain the adapter and generic schema, not personal embedding vectors or audio.

I do not export a candidate until I have completed validation-based selection and a final independent test. Even after passing statistical tests, production use requires additional real-world testing; this event prototype is not a certified medical or assistive device.

In [ ]:
import joblib

def export_frozen_recognizer(name,chosen_artifact,encoder_name=CONFIG['encoder_id']):
    if 'policy' not in chosen_artifact or 'model' not in chosen_artifact:raise ValueError('Unvalidated model')
    if not np.isfinite(chosen_artifact['policy']['threshold']):
        raise ValueError('Invalid acceptance threshold')
    meta={'name':name,'encoder_id':encoder_name,'labels':chosen_artifact['classes'].tolist(),
          'threshold':float(chosen_artifact['policy']['threshold']),
          'sample_rate':16000,'canonical_utterances':{k:v['canonical'] for k,v in COMMANDS.items()},
          'calibration':chosen_artifact['policy'],'run_id':RUN_ID}
    location=ARTIFACTS/f'{name}.joblib'
    joblib.dump({'model':chosen_artifact['model'],'meta':meta},location)
    (ARTIFACTS/f'{name}.json').write_text(json.dumps(meta,indent=2))
    print('PRIVATE deployment model:',location)
    return location

if RUN_FROZEN_EXPERIMENT:
    deployment_artifact=export_frozen_recognizer('team4_personalized',chosen)
else:
    print('Export blocked until a genuine validation and trained model exist.')

## 15. Toolkit integration: exact contract verified in the public repository

I verified the following interfaces in the official repository on 2 November 2026:

- `controller/web/ui.js`: default dictation uses browser `SpeechRecognition` with `recog.lang = 'en-US'` and sends recognized text to `submit()`.
- `controller/createController.js`: `createController({control, ...})` returns a controller whose `handle(utterance)` performs grammar resolution, confirmation as appropriate, and dispatch.
- `controller/grammar.js`: recognized canonical strings **`scroll down`**, **`scroll up`**, **`go back`**, **`read this`**, **`undo`** have deterministic mappings.
- `controller/web/dom-receiver.js`: `createDomReceiver(root)` creates a real `ControlPort` that can scroll, activate, navigate back/forward, read content and apply settings.
- `controller/PROTOCOL.md`: remote transport can be introduced later, but this prototype does **not** invent a nonstandard wire format or call LLM agents for high-risk commands.

**Architectural decision:** a *sidecar companion interface* uses browser audio capture → Python personalized inference → `UNKNOWN` or a vetted **canonical utterance** → `controller.handle(canonical)` → official `ControlPort`. This is a **supported composition of the headless Controller**, not a claim that its default speech widget already exposes an injection hook. We do not secretly patch upstream code. If we later propose a native speech-provider hook upstream, it should be a separate minimal pull request with tests.

Before deployment I confirm the upstream commit SHA, the export names, the browser origin and all tests. I never bypass confirmation prompts: if the Controller returns `pending`, the co-designer must explicitly confirm using an accessible independent control. The first demo uses low-risk navigation commands only.

## 16. Python inference API design and safeguards

I implement a local, single-purpose inference endpoint. It accepts one bounded PCM WAV recording, resamples, applies the **exact same** feature extractor as training, runs the private classifier, and returns only the approved `command_id`, canonical utterance, acceptance status and a confidence-related score. I keep the service on loopback, disable audio logging, limit request size, and refuse actions when artifacts are absent. The browser UI is separately served on localhost by the official toolkit demo server; explicit CORS is needed for that localhost origin, not a wildcard.

A personal voice is biometric-like sensitive information. This API is a hackathon prototype, not hardened infrastructure suitable for exposure to the Internet or processing sensitive interactions.

In [ ]:
# This cell writes an inspectable FastAPI prototype, NOT a deployed public endpoint.
API_SOURCE = r'''from pathlib import Path
from io import BytesIO
from functools import lru_cache
import os, joblib, numpy as np, soundfile as sf
from scipy.signal import resample_poly
from math import gcd
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware

MODEL_PATH = Path(os.environ.get('TEAM4_MODEL_PATH','team4_voice_lab/private/models/team4_personalized.joblib'))
MAX_BYTES = 2_000_000
app = FastAPI(title='Team4 Local Personalized Speech Adapter',docs_url=None,redoc_url=None)
app.add_middleware(CORSMiddleware,allow_origins=['http://127.0.0.1:4100','http://localhost:4100'],
                   allow_credentials=False,allow_methods=['POST','GET'],allow_headers=['content-type'])

@lru_cache(maxsize=1)
def resources():
    if not MODEL_PATH.is_file():raise RuntimeError('A validated personal model is not installed')
    from transformers import AutoFeatureExtractor, AutoModel
    import torch
    bundle=joblib.load(MODEL_PATH)  # trusted local artifact only, never upload user files
    meta=bundle['meta'];encoder_id=meta['encoder_id']
    extractor=AutoFeatureExtractor.from_pretrained(encoder_id)
    encoder=AutoModel.from_pretrained(encoder_id).eval()
    return bundle,extractor,encoder

@app.get('/health')
def health():return {'state':'ready' if MODEL_PATH.is_file() else 'not_enrolled',
                      'model_installed':MODEL_PATH.is_file()}

@app.post('/recognize')
async def recognize(audio: UploadFile=File(...)):
    # Fail closed; audio never written to disk or printed in logs.
    if not MODEL_PATH.exists():raise HTTPException(status_code=503,detail='Enrollment not completed')
    payload=await audio.read(MAX_BYTES+1)
    if len(payload)>MAX_BYTES:raise HTTPException(status_code=413,detail='Audio too large')
    try:
        waveform,sr=sf.read(BytesIO(payload),dtype='float32',always_2d=True)
        x=waveform.mean(axis=1)
        if sr<8000 or sr>96000:raise ValueError('Unreasonable sample rate')
        if sr!=16000:
            g=gcd(sr,16000);x=resample_poly(x,16000//g,sr//g).astype('float32')
        if not .25<=len(x)/16000<=10:raise ValueError('Invalid clip duration')
        if not np.isfinite(x).all() or np.sqrt(np.mean(x*x))<.002:
            return {'accepted':False,'command_id':'UNKNOWN','canonical':None,'reason':'silent_or_invalid'}
        bundle,extractor,encoder=resources()
        import torch
        inputs=extractor([x],sampling_rate=16000,padding=True,return_attention_mask=True,return_tensors='pt')
        with torch.inference_mode():
            h=encoder(**inputs).last_hidden_state
            mask=encoder._get_feature_vector_attention_mask(h.shape[1],inputs['attention_mask'])
            vec=(h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp(min=1)
            vec=torch.nn.functional.normalize(vec,p=2,dim=-1).cpu().numpy()
        model=bundle['model'];meta=bundle['meta'];scores=model.predict_scores(vec)
        labels=np.array(model.classes_);select=labels!='UNKNOWN'
        candidates=labels[select]; vals=scores[:,select]
        top=int(vals.argmax(axis=1)[0]);best=float(vals[0,top]);label=str(candidates[top])
        accepted=bool(best>=float(meta['threshold']))
        # Score is not a calibrated probability.
        return {'accepted':accepted,'command_id':label if accepted else 'UNKNOWN',
                'canonical':meta['canonical_utterances'].get(label) if accepted else None,
                'score':round(best,5),'reason':'accepted' if accepted else 'below_threshold'}
    except Exception as exc:
        # Never disclose raw audio, path, private text or full traceback through HTTP.
        raise HTTPException(status_code=422,detail='Invalid or unsupported WAV input') from None
'''
API_PATH = ROOT / 'team4_api.py'
API_PATH.write_text(API_SOURCE)
print('Created API scaffold:',API_PATH)
print('Run locally AFTER trained artifact exists: uvicorn team4_api:app --host 127.0.0.1 --port 8765')

## 17. Browser capture and ControlPort integration

I use the browser's Web Audio `AudioWorklet` to capture PCM float audio from a microphone activated through an explicit button, then encode WAV locally and send it to loopback. The UI shows listening state and has independent Stop/Cancel buttons; background audio capture is not permitted. The audio buffer is ephemeral in JavaScript and discarded after inference. A keyboard path remains available. No page is allowed to autoexecute an unrecognized utterance.

**Browser prerequisite:** serve through `http://localhost:4100` or HTTPS (secure context for microphone permission). The backend binds only to `127.0.0.1:8765` and uses narrow CORS. Users must grant mic permission. For screen reader friendliness I use an ARIA live region, and the Controller's own feedback continues to govern actual app actions.

In [ ]:
WORKLET_JS = r'''class Team4PCMProcessor extends AudioWorkletProcessor {
  process(inputs) {
    const channel = inputs[0] && inputs[0][0];
    if (channel) this.port.postMessage(Float32Array.from(channel));
    return true;
  }
}
registerProcessor('team4-pcm-recorder', Team4PCMProcessor);
'''

BRIDGE_JS = r'''// This sidecar uses the documented headless Controller, NOT a private upstream hook.
import { createController } from '/controller/createController.js';
import { createDomReceiver } from '/controller/web/dom-receiver.js';
const appRoot = document.querySelector('#target-app');
const controller = createController({ control: createDomReceiver(appRoot) });
const status = document.querySelector('#status');
const start = document.querySelector('#listen');
const cancel = document.querySelector('#cancel');
const manual = document.querySelector('#manual');
const manualSubmit = document.querySelector('#manual-submit');
let ctx, stream, source, worker, silentGain, captureTimer, chunks=[], active=false;
const allowed = new Set(['SCROLL_DOWN','SCROLL_UP','GO_BACK','READ_PAGE','UNDO']);
function say(s){status.textContent=String(s);}
function toWav(frames,sampleRate){
  const count=frames.reduce((n,x)=>n+x.length,0),buffer=new ArrayBuffer(44+2*count);
  const v=new DataView(buffer);let i=0;
  function chars(s){for(const c of s)v.setUint8(i++,c.charCodeAt(0));}
  function u16(n){v.setUint16(i,n,true);i+=2;}
  function u32(n){v.setUint32(i,n,true);i+=4;}
  chars('RIFF');u32(36+2*count);chars('WAVEfmt ');u32(16);u16(1);u16(1);
  u32(sampleRate);u32(sampleRate*2);u16(2);u16(16);chars('data');u32(count*2);
  for(const frame of frames){for(const val of frame){
    const s=Math.max(-1,Math.min(1,val));v.setInt16(i,s<0?s*32768:s*32767,true);i+=2;
  }}
  return new Blob([buffer],{type:'audio/wav'});
}
async function cleanup(){
  if(captureTimer)clearTimeout(captureTimer);captureTimer=null;
  if(worker){try{worker.disconnect();worker.port.onmessage=null;}catch{}} worker=null;
  if(silentGain){try{silentGain.disconnect();}catch{}}silentGain=null;
  if(source){try{source.disconnect();}catch{}}source=null;
  if(stream){stream.getTracks().forEach(t=>t.stop());}stream=null;
  if(ctx){try{await ctx.close();}catch{}}ctx=null;
  active=false;start.disabled=false;cancel.disabled=true;
}
start.addEventListener('click',async()=>{
  if(active)return;
  chunks=[];start.disabled=true;cancel.disabled=false;
  try{
    stream=await navigator.mediaDevices.getUserMedia({audio:{echoCancellation:true,noiseSuppression:false},video:false});
    ctx=new AudioContext();await ctx.audioWorklet.addModule('./pcm-worklet.js');
    source=ctx.createMediaStreamSource(stream);
    worker=new AudioWorkletNode(ctx,'team4-pcm-recorder');
    worker.port.onmessage=e=>chunks.push(e.data);
    source.connect(worker); silentGain=ctx.createGain();silentGain.gain.value=0;
    worker.connect(silentGain);silentGain.connect(ctx.destination);
    active=true;captureTimer=setTimeout(()=>{if(active)finish();},8000);say('Listening. Use Stop and evaluate to finish, or Cancel to discard.');
  }catch(e){say('Microphone unavailable. Use keyboard controls.');await cleanup();}
});
async function finish(){
  if(!active)return;
  const recorded=toWav(chunks,ctx.sampleRate);
  await cleanup();chunks=[];
  if(recorded.size>2_000_000){say('Recording too long. Please try a shorter command.');return;}
  say('Processing locally. No action has been executed yet.');
  const body=new FormData();body.append('audio',recorded,'command.wav');
  try{
    const response=await fetch('http://127.0.0.1:8765/recognize',{method:'POST',body});
    if(!response.ok)throw new Error('Unavailable or invalid input');
    const result=await response.json();
    if(!result.accepted || !allowed.has(result.command_id)||!result.canonical){say('I did not recognize a safe command. Please retry or use the keyboard.');return;}
    // No free-form LLM / no unverified command string passed to the Controller.
    say('Heard: '+result.canonical+'. Executing through AI4A11y Controller.');
    const outcome=await controller.handle(result.canonical);
    say(outcome.pending ? 'Confirmation required. Use keyboard confirmation.' : (outcome.say||'Action completed.'));
  }catch(e){say('Recognition is unavailable. No voice command was performed.');}
}
document.querySelector('#finish').addEventListener('click',finish);
cancel.addEventListener('click',async()=>{await cleanup();chunks=[];say('Recording discarded.');});
manualSubmit.addEventListener('click',async()=>{
  const value=manual.value.trim();
  if(!value)return;
  const outcome=await controller.handle(value);say(outcome.say || 'No action performed.');
});
'''

DEMO_HTML = r'''<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Team 4 Accessible Voice Controller</title>
<style>body{font:1.15rem system-ui;max-width:58rem;margin:2rem auto;padding:1rem;line-height:1.6}
button,input{font:inherit;padding:.65rem;margin:.3rem;min-height:48px}button:focus-visible,input:focus-visible{outline:4px solid #225ad7}
#target-app{border:2px solid #aaa;padding:1rem;max-height:15rem;overflow:auto}
#status{background:#f5f5f5;color:#111;padding:1rem;min-height:2rem}</style></head><body>
<h1>A Voice That Understands Me</h1>
<p>Local experimental prototype. Speak or use the keyboard. Never use for sensitive transactions.</p>
<button id="listen">Start listening</button><button id="finish">Stop and evaluate</button>
<button id="cancel" disabled>Cancel recording</button>
<p id="status" role="status" aria-live="polite">Ready. Microphone remains off until activated.</p>
<label for="manual">Keyboard alternative</label><input id="manual" type="text" autocomplete="off">
<button id="manual-submit">Submit typed command</button>
<section id="target-app" tabindex="0" aria-label="Target application demo">
<h2>Accessible navigation practice</h2><p>This content can be read, scrolled, and adapted.</p>
<p>Section one. I can practice voice control using my own speech.</p>
<p>Section two. This application uses the official AI4A11y Controller.</p>
<p>Section three. I can opt out and use the keyboard.</p>
</section><script type="module" src="./bridge.mjs"></script></body></html>'''

# The demo lives inside the toolkit root so module import paths resolve unchanged.
for name,content in [('pcm-worklet.js',WORKLET_JS),('bridge.mjs',BRIDGE_JS),('index.html',DEMO_HTML)]:
    out=ROOT/'toolkit_sidecar'/name;out.parent.mkdir(parents=True,exist_ok=True);out.write_text(content)
print('Generated integration sidecar:',ROOT/'toolkit_sidecar')

### Integration smoke test and a critical caveat

I will copy `toolkit_sidecar/` into the cloned **toolkit root**, preserving `/controller/` paths, serve the **toolkit root** at port 4100, and open `/toolkit_sidecar/index.html`. The demo deliberately uses **low-risk commands only**. Its typed input is an accessible backup.

The web demo's `AudioWorklet` recorder does not yet implement voice activity detection, automatic endpointing, a press-free start mechanism or a hard recording timer. Those are co-design requirements to evaluate, not features I will claim are finished. In deployment I would replace this prototype with a more mature capture method selected with the user. Some browsers require the worklet node to be connected to the audio graph for continuous processing; this must be browser-tested before presentation. Do **not** claim browser compatibility before testing the actual device.

**Safety note:** The currently generated demo uses `controller.handle` for the five verified canonical phrases. It does not send prediction labels directly to `performAction`, which would bypass the official grammar and confirmation layer.

### 17A. Node.js conformance checks after copying sidecar

I use the toolkit's existing ESM exports and confirm them at a **pinned commit** before the final demo. If the upstream repository evolves and the import paths change, I modify the adapter according to actual exports; I do not silently use an undocumented SDK. On the presentation laptop I run the toolkit's `npm test` and a browser test with the co-designer. The following commands are performed in a terminal, not executed automatically by this notebook:

```bash
cd toolkit-upstream
node --version
npm test
mkdir -p toolkit_sidecar
# Copy the three generated browser files from team4_voice_lab/toolkit_sidecar here.
python -m http.server 4100
# Second terminal, in team4_voice_lab with correct Python environment and PRIVATE model:
uvicorn team4_api:app --host 127.0.0.1 --port 8765
# Open http://127.0.0.1:4100/toolkit_sidecar/index.html
```

The mic and server run only on local machine. Public deployment, user authentication, encrypted retention, rate limiting and production safeguarding are **out of scope for this experimental scaffold**, and I must not expose port 8765 publicly.

## 18. Automated integrity tests (can run without model weights)

I verify data semantics and policy behavior without needing to download any restricted data. Synthetic waveform tests test **software mechanics**, not the person's speech. I deliberately do not present synthetic classification scores as research outcomes.

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as tmp:
    a=Path(tmp)/'tone.wav'
    sr=8000;t=np.arange(sr)/sr
    sf.write(a,.2*np.sin(2*np.pi*350*t),sr)
    x,sr2=read_audio(a)
    assert sr2==16000 and len(x)==16000
    q=audio_qa(a)
    assert not q['is_silent'] and .95<q['duration_s']<1.05
# Test safety: malicious or unrelated transcript is not authorized.
assert map_transcript_to_command('transfer funds now','en-KE')==NEGATIVE
assert map_transcript_to_command('send personal data','sw-KE')==NEGATIVE
# Test independent sessions.
fixture=pd.DataFrame([dict(zip(COLUMNS,[str(Path('a.wav').resolve()),'personA','session1','en-KE','UNDO',True,True,'consented','research','train','2026-11-02'])),
                     dict(zip(COLUMNS,[str(Path('b.wav').resolve()),'personA','session2','en-KE','UNDO',True,True,'consented','research','val','2026-11-03'])),
                     dict(zip(COLUMNS,[str(Path('c.wav').resolve()),'personA','session3','en-KE','UNKNOWN',True,True,'consented','research','test','2026-11-04']))])
assert assert_split_integrity(fixture)['status']=='pass'
try:
    bad=fixture.copy();bad.loc[1,'session_id']='session1';assert_split_integrity(bad)
    raise AssertionError('Leakage test failed to detect overlapping sessions')
except AssertionError as exc:
    if 'Leakage test' in str(exc):raise
print('PASS: resampling, QA, deny-by-default, session isolation')

## 19. Submission and reporting checklist

I will only claim **implemented** what passes a real verification. The organizers require a public repository or toolkit pull request, README and writeup, short demo video, and toolkit issues from co-designer testing. I must demonstrate with the co-designer, not use them as a test subject without decision making power.

Before the hackathon I should (a) request CDLI corpus access, (b) ask organizers whether the datasets can be used during the event and whether recording and voice model storage are permitted, (c) ask the co-designer which five actions help, (d) confirm their recording and app interaction preferences, (e) install Node 20+, verify the toolkit version and its tests, and (f) agree on evaluation responsibilities. For the final handoff I export only nonidentifying results and list all limitations.

**My report template:** scope and co-design decisions; data source, version, access basis and consent; exact split protocol; baseline and personalized models; hyperparameters and hardware; unknown speech policy and calibration; held-out per-language metrics with denominators and uncertainty; qualitative task observations; privacy and deletion protocol; toolkit code change or issue links; short demo; honest unresolved limitations.

In [ ]:
submission = {
  'team':'Problem 4 | A Voice That Understands Me',
  'deliverables':{'working_code':False,'readme_and_writeup':False,'demo_video':False,'toolkit_issues':False},
  'verification':{'co_designer_consented':False,'train_val_test_separate_sessions':False,
     'english_and_swahili_evaluated':False,'unknown_speech_tested':False,'toolkit_actions_tested':False,
     'browser_and_device_tested':False,'private_model_not_committed':False},
  'status':'RESEARCH BLUEPRINT ONLY — no co-designer experiments or outcomes claimed'
}
(PUBLIC/'submission_checklist.json').write_text(json.dumps(submission,indent=2))
print(json.dumps(submission,indent=2))

## 20. Source register and limitations

**Official toolkit:** [Repository](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit), [Controller API](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit/blob/main/controller/createController.js), [Grammar](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit/blob/main/controller/grammar.js), [DOM receiver](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit/blob/main/controller/web/dom-receiver.js), [Voice UI](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit/blob/main/controller/web/ui.js), [Protocol](https://github.com/AI-for-Accessibility-Collective/AI-for-Accessibility-Toolkit/blob/main/controller/PROTOCOL.md). Source inspected via GitHub connector on 2026-11-02; the live repository may evolve, so pin the commit before running the app.

**Data:** [CDLI Kenyan English](https://huggingface.co/datasets/cdli/kenyan_english_nonstandard_speech_v1.0), [CDLI Kenyan Swahili](https://huggingface.co/datasets/cdli/kenyan_swahili_nonstandard_speech_v1.0), [CDLI model English](https://huggingface.co/cdli/whisper-small_finetuned_kenyan_english_nonstandard_speech_v1.0), [CDLI model Swahili](https://huggingface.co/cdli/whisper-small_finetuned_kenyan_swahili_nonstandard_speech_v1.0), [AfriSpeech 200](https://huggingface.co/datasets/intronhealth/afrispeech-200), [FLEURS](https://huggingface.co/datasets/google/fleurs), [ALFFA](https://www.openslr.org/25/), [Mozilla Data Collective](https://mozilladatacollective.com/).

**Academic:** Yee et al. *Latent Phrase Matching for Dysarthric Speech*, Interspeech 2023. Müller-Eberstein and colleagues, atypical-speech adaptation research, TACL 2024. Raja and colleagues, *Idiosyncratic Versus Normative Modeling of Atypical Speech Recognition*, EMNLP 2025.

**Limitations:** The two most relevant corpora are gated; no downloaded recordings are bundled. No participant voice has been enrolled; no performance measurements have been obtained. Baseline model cards may contain pretrained dataset overlap, so unrelated published WER must not be interpreted as an independent test here. Realtime streaming, automated endpointing, model calibration under extreme scarcity and production data security require further implementation and validation. The Python API and browser integration are proof-of-concept scaffolds until exercised with the pinned upstream repository and tested on the actual machine and browser. The notebook does not claim clinical suitability.

**Final principle:** I am trying to give one person genuine agency. A reliable, reversible five-command control experience is a stronger outcome than an unreliable general speech transcription system.